# Clone a narration voice — free GPU

Drains `video_voice_queue` into the render cache. Runs on **Kaggle** (30 GPU-hours/week,
published quota) or **Colab** (~15–30, dynamic). A five-minute video is roughly 2–3 minutes of
GPU, so the free quota is hundreds of videos.

**Before you run this**
1. `npm run voice:clone -- --storyboard=<uuid> --queue` on your machine.
2. Turn the GPU on — Kaggle: *Settings → Accelerator → GPU T4 ×2*. Colab: *Runtime → Change
   runtime type → T4*.
3. Add the four secrets named in cell 2. **Never paste a secret into a cell** — this notebook
   lives in a public repo.

**What it must not do**
- It must never compute a cache key. `text_hash` comes from the queue, written by the same
  TypeScript the renderer looks up with. JS renders `${1.0}` as `1` and Python gives `1.0`, so a
  recomputed key silently misses every clip.
- It must never synthesize Japanese. A voice cloned from English or Hindi reading Japanese is
  confident mispronunciation — the worst failure a teaching video has. Cell 4 refuses.


In [ ]:
# 1. Dependencies. ~3 minutes on a cold runtime.
!pip install -q chatterbox-tts psycopg2-binary boto3

import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings > Accelerator > GPU. Colab: Runtime > Change runtime type."


In [ ]:
# 2. Secrets. Read from the platform's secret store — never written into this cell.
#
# Create these four with EXACTLY these names:
#   VOICE_DATABASE_URL   the scoped Neon role (see docs/VIDEO_MANUAL_STEPS.md), not your main one
#   R2_ENDPOINT          https://<account>.r2.cloudflarestorage.com
#   R2_ACCESS_KEY_ID     token scoped to the bucket
#   R2_SECRET_ACCESS_KEY
#   R2_BUCKET_NAME       and R2_BUCKET_URL for the public prefix
#
# Kaggle: Add-ons > Secrets.   Colab: the key icon in the left sidebar.
import os

def secret(name, required=True):
    try:                                   # Kaggle
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:                                   # Colab
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    v = os.environ.get(name)               # anywhere else
    if not v and required:
        raise RuntimeError(f"{name} is not set. Add it to the notebook's secret store.")
    return v

DATABASE_URL     = secret("VOICE_DATABASE_URL")
R2_ENDPOINT      = secret("R2_ENDPOINT")
R2_ACCESS_KEY_ID = secret("R2_ACCESS_KEY_ID")
R2_SECRET_KEY    = secret("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME   = secret("R2_BUCKET_NAME")
R2_BUCKET_URL    = secret("R2_BUCKET_URL")
print("secrets loaded")


In [ ]:
# 3. Reference audio — the ~60s samples of your voice.
#
# Pasted as pre-signed R2 URLs valid for an hour, so your voice sample is never committed to a
# public repo and no long-lived credential is needed to fetch it. Generate them with:
#   aws s3 presign s3://$R2_BUCKET_NAME/reference/avnish-en.wav --expires-in 3600 --endpoint-url $R2_ENDPOINT
import urllib.request, pathlib

REFERENCE_URLS = {
    "en": "",   # <- paste the pre-signed URL for avnish-en.wav
    "hi": "",   # <- paste the pre-signed URL for avnish-hi.wav
}

pathlib.Path("reference").mkdir(exist_ok=True)
REFERENCE_PATHS = {}
for lang, url in REFERENCE_URLS.items():
    if not url:
        print(f"  {lang}: no URL given — {lang} lines will be skipped")
        continue
    dest = f"reference/{lang}.wav"
    urllib.request.urlretrieve(url, dest)
    size = pathlib.Path(dest).stat().st_size
    # A pre-signed URL that has expired returns a short XML error body, not audio. Catching it
    # here beats discovering it as a mysteriously bad clone.
    assert size > 20_000, f"{dest} is {size} bytes — the URL probably expired. Re-presign it."
    REFERENCE_PATHS[lang] = dest
    print(f"  {lang}: {size/1000:.0f} KB")


In [ ]:
# 4. What needs generating.
import psycopg2, psycopg2.extras

conn = psycopg2.connect(DATABASE_URL)
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT id, text_hash, segment_id, text, lang, voice_name
        FROM video_voice_queue
        WHERE status = 'pending'
        ORDER BY created_at
    """)
    pending = cur.fetchall()

# Defence in depth. voiceForSegment() already forces a native ja-JP voice, so a ja row here means
# something upstream changed and a video is about to get mispronounced Japanese.
ja = [r for r in pending if r["lang"] == "ja"]
assert not ja, f"{len(ja)} Japanese row(s) queued. Japanese must never be cloned — stop and fix the caller."

missing_ref = [r for r in pending if r["lang"] not in REFERENCE_PATHS]
if missing_ref:
    print(f"  {len(missing_ref)} row(s) have no reference audio for their language and will be skipped")

work = [r for r in pending if r["lang"] in REFERENCE_PATHS]
print(f"{len(pending)} pending, {len(work)} generatable")
for r in work[:5]:
    print("   ", r["lang"], r["text"][:70])


In [ ]:
# 5. Generate, upload, record. Safe to re-run — finished rows leave the queue.
#
# TWO INVARIANTS:
#   - the R2 key is exactly video/tts/<text_hash>.wav, which is what the renderer looks up
#   - text_hash comes from the queue and is never recomputed here
import io, wave, boto3, torch
from chatterbox.tts import ChatterboxMultilingualTTS

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
s3 = boto3.client("s3", endpoint_url=R2_ENDPOINT,
                  aws_access_key_id=R2_ACCESS_KEY_ID, aws_secret_access_key=R2_SECRET_KEY)

def wav_bytes(tensor, sample_rate):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sample_rate)
        w.writeframes((tensor.squeeze().cpu().numpy() * 32767).astype("int16").tobytes())
    return buf.getvalue()

done = failed = 0
for i, row in enumerate(work, 1):
    try:
        audio = model.generate(row["text"], language_id=row["lang"],
                               audio_prompt_path=REFERENCE_PATHS[row["lang"]])
        data = wav_bytes(audio, model.sr)

        # Duration read from the RIFF header, the same way the renderer measures a Google clip —
        # every scene's frame span is computed from this number.
        with wave.open(io.BytesIO(data)) as w:
            duration = w.getnframes() / float(w.getframerate())

        key = f"video/tts/{row['text_hash']}.wav"
        s3.put_object(Bucket=R2_BUCKET_NAME, Key=key, Body=data, ContentType="audio/wav")
        url = f"{R2_BUCKET_URL.rstrip('/')}/{key}"

        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO video_tts_assets
                  (text_hash, lang_code, voice_name, speaking_rate, pitch, text_preview,
                   audio_url, duration_seconds, char_count, estimated_cost_usd)
                SELECT q.text_hash, q.lang, q.voice_name, q.speaking_rate, q.pitch,
                       left(q.text, 200), %s, %s, length(q.text), 0
                FROM video_voice_queue q WHERE q.id = %s
                ON CONFLICT (text_hash) DO NOTHING
            """, (url, duration, row["id"]))
            cur.execute("UPDATE video_voice_queue SET status='done', claimed_at=NOW() WHERE id=%s", (row["id"],))
        conn.commit()
        done += 1
        print(f"  [{i}/{len(work)}] {duration:5.1f}s  {row['text'][:56]}")
    except Exception as e:
        conn.rollback()
        with conn.cursor() as cur:
            cur.execute("UPDATE video_voice_queue SET status='failed', error_message=%s WHERE id=%s",
                        (str(e)[:500], row["id"]))
        conn.commit()
        failed += 1
        print(f"  [{i}/{len(work)}] FAILED  {str(e)[:90]}")

print(f"\n{done} generated, {failed} failed")


In [ ]:
# 6. Where things stand.
with conn.cursor() as cur:
    cur.execute("SELECT status, COUNT(*) FROM video_voice_queue GROUP BY status ORDER BY status")
    for status, n in cur.fetchall():
        print(f"  {status:8} {n}")
conn.close()

print("""
Done. Back on your machine:
  - re-render the project; the worker finds every clip in the cache and loads no model
  - anything 'failed' keeps its error_message — fix and re-run this notebook
  - a session that died part-way is fine: 'done' rows stay done
""")
